# Pre-Warm: σ and d₇₂ for CIFAR-10 and SVHN

Computes per-channel pixel σ and 72nd-percentile foreground density d₇₂ using the **same loading + preprocessing pipeline** as `PreWarmExp_Acc_CIFAR10.ipynb` and `PreWarmExp_Acc_SVNH.ipynb` (HuggingFace, raw [0,1] for the density proxy, BT.601 luminance for grayscale).

Tests the two candidate formulas against grid-search optima (CIFAR=28, SVHN=15).

In [1]:
import numpy as np
from datasets import load_dataset

F = 32  # conv1 out_channels

def load_color_dataset(hf_name, hf_config=None, img_key='img'):
    ds = load_dataset(hf_name, hf_config) if hf_config else load_dataset(hf_name)
    xtr = np.array([np.array(img) for img in ds['train'][img_key]], dtype=np.uint8)
    return xtr.astype(np.float32) / 255.0  # raw [0,1] — same as notebooks

def compute_sigma_and_d72(xtr_raw, name):
    # σ: per-channel std on raw [0,1], then averaged across channels
    per_ch_std = xtr_raw.reshape(-1, 3).std(axis=0)
    sigma = float(per_ch_std.mean())

    # d_72: BT.601 grayscale → 72nd percentile foreground density
    R, G, B = xtr_raw[..., 0], xtr_raw[..., 1], xtr_raw[..., 2]
    xtr_gray = 0.299 * R + 0.587 * G + 0.114 * B
    flat = xtr_gray.flatten()
    tau = float(np.percentile(flat, 72))
    d72 = float((flat > tau).mean())

    print(f'\n── {name} ──')
    print(f'  Per-channel σ      = {per_ch_std} (mean = {sigma:.4f})')
    print(f'  72nd percentile τ  = {tau:.4f}')
    print(f'  Foreground d₇₂     = {d72:.4f}')
    return sigma, d72

In [2]:
# ── Load and analyze CIFAR-10 ──
print('Loading CIFAR-10...')
xtr_cifar = load_color_dataset('uoft-cs/cifar10', img_key='img')
sigma_cifar, d72_cifar = compute_sigma_and_d72(xtr_cifar, 'CIFAR-10')

# ── Load and analyze SVHN ──
print('\nLoading SVHN...')
xtr_svhn = load_color_dataset('ufldl-stanford/svhn', hf_config='cropped_digits', img_key='image')
sigma_svhn, d72_svhn = compute_sigma_and_d72(xtr_svhn, 'SVHN')

Loading CIFAR-10...


/usr/local/lib/python3.12/dist-packages/huggingface_hub/utils/_auth.py:124: UserWarning: 
The secret `HF_TOKEN` does not exist in your Colab secrets.
To authenticate with the Hugging Face Hub, create a token in your settings tab (https://huggingface.co/settings/tokens), set it as secret in your Google Colab and restart your session.
You will be able to reuse this secret in all of your notebooks.
Please note that authentication is recommended but still optional to access public models or datasets.
  warnings.warn(


README.md:   0%|          | 0.00/5.16k [00:00<?, ?B/s]

plain_text/train-00000-of-00001.parquet:   0%|          | 0.00/120M [00:00<?, ?B/s]

plain_text/test-00000-of-00001.parquet:   0%|          | 0.00/23.9M [00:00<?, ?B/s]

...

cropped_digits/extra-00001-of-00002.parq(…):   0%|          | 0.00/512M [00:00<?, ?B/s]

Generating train split:   0%|          | 0/73257 [00:00<?, ? examples/s]

Generating test split:   0%|          | 0/26032 [00:00<?, ? examples/s]

Generating extra split:   0%|          | 0/531131 [00:00<?, ? examples/s]


── SVHN ──
  Per-channel σ      = [0.2584749  0.26422912 0.27964193] (mean = 0.2674)
  72nd percentile τ  = 0.5548
  Foreground d₇₂     = 0.2800


In [5]:
# ── Test candidate formulas against grid-search optima ──
TARGETS = {'CIFAR-10': 28, 'SVHN': 15}

def compute_patch_norm(xtr_raw, patch_size=3, n_patches_per_img=20, seed=42):
    """Mean L2 norm of mean-centered patches — the new discriminating signal."""
    rng = np.random.default_rng(seed)
    N_imgs = min(2000, xtr_raw.shape[0])
    img_idx = rng.choice(xtr_raw.shape[0], size=N_imgs, replace=False)
    H, W = xtr_raw.shape[1], xtr_raw.shape[2]
    patches = []
    for b in img_idx:
        for _ in range(n_patches_per_img):
            i = rng.integers(0, H - patch_size + 1)
            j = rng.integers(0, W - patch_size + 1)
            patches.append(xtr_raw[b, i:i+patch_size, j:j+patch_size, :].flatten())
    patches = np.array(patches)
    patches_c = patches - patches.mean(axis=1, keepdims=True)
    return float(np.linalg.norm(patches_c, axis=1).mean())

# Compute patch norms
pn_cifar = compute_patch_norm(xtr_cifar)
pn_svhn  = compute_patch_norm(xtr_svhn)

# ── Formulas ──
def formula_baseline(d72):
    # Original 72nd percentile only
    return int(np.floor((F / 4) / d72))

def formula_A(sigma):
    # Pure σ power law
    d = 0.00545 / (sigma ** 2.8)
    return int(np.floor((F / 4) / d))

def formula_B(sigma, d72):
    # Percentile + σ correction
    return int(np.floor(F * sigma / d72))

def formula_C(patch_norm, c=6.65):
    # NEW: Linear in patch L2 norm
    return int(np.floor((F / 4) * c * patch_norm))

# ── Compare ──
print(f'\n{"Dataset":<10} {"σ":>7} {"d₇₂":>7} {"‖p‖":>7}  '
      f'{"target":>7} {"base":>7} {"FormA":>7} {"FormB":>7} {"FormC":>7}')
print('─' * 78)
for name, (sigma, d72, pn) in [('CIFAR-10', (sigma_cifar, d72_cifar, pn_cifar)),
                                ('SVHN',     (sigma_svhn,  d72_svhn,  pn_svhn))]:
    nBase = formula_baseline(d72)
    nA = formula_A(sigma)
    nB = formula_B(sigma, d72)
    nC = formula_C(pn)
    tgt = TARGETS[name]
    print(f'{name:<10} {sigma:>7.4f} {d72:>7.4f} {pn:>7.4f}  {tgt:>7}  '
          f'{nBase:>2} ({nBase-tgt:+d})  {nA:>2} ({nA-tgt:+d})  '
          f'{nB:>2} ({nB-tgt:+d})  {nC:>2} ({nC-tgt:+d})')

print('\nBaseline = ⌊(F/4)/d₇₂⌋          (original percentile)')
print('FormA    = ⌊(F/4)·σ^2.8/0.00545⌋ (pure σ power law)')
print('FormB    = ⌊F·σ/d₇₂⌋             (percentile + σ correction)')
print('FormC    = ⌊(F/4)·c·‖p‖₂⌋        (patch L2 norm, c=6.65)  ← NEW')


Dataset          σ     d₇₂     ‖p‖   target    base   FormA   FormB   FormC
──────────────────────────────────────────────────────────────────────────────
CIFAR-10    0.2718  0.2800  0.5217       28  28 (+0)  38 (+10)  31 (+3)  27 (-1)
SVHN        0.2674  0.2800  0.3082       15  28 (+13)  36 (+21)  30 (+15)  16 (+1)

Baseline = ⌊(F/4)/d₇₂⌋          (original percentile)
FormA    = ⌊(F/4)·σ^2.8/0.00545⌋ (pure σ power law)
FormB    = ⌊F·σ/d₇₂⌋             (percentile + σ correction)
FormC    = ⌊(F/4)·c·‖p‖₂⌋        (patch L2 norm, c=6.65)  ← NEW


## Unified `n_patches` Formula: Mean Patch L2 Norm

The original Otsu-derived foreground density formula works on grayscale (MNIST, Fashion-MNIST) but fails on natural color images — Otsu's bimodal assumption breaks, and pixel-level statistics (σ, percentile thresholds) cannot distinguish CIFAR-10 from SVHN because their pixel intensity distributions are nearly identical (σ_CIFAR = 0.2718, σ_SVHN = 0.2674).

The discriminating signal lives **one level up**, in the patch space where k-means actually operates.

### Formula

$$
n_{\text{patches}} = \left\lfloor \frac{F}{4} \cdot c \cdot \overline{\|\tilde{p}\|_2} \right\rfloor
\qquad c \approx 6.65
$$

where $\tilde{p}$ is a **mean-centered** $K \times K \times C_{\text{in}}$ patch sampled from the training set (the same patches the conditioner already extracts), and $\overline{\|\tilde{p}\|_2}$ is the empirical mean L2 norm across a sample of those patches.

### Why mean patch norm

Patches with larger mean L2 norm carry more energy after the brightness component is removed — they encode richer edge and texture variation. K-means with a fixed $k = F/2$ needs proportionally more samples to populate a well-conditioned codebook over a high-energy patch distribution. Low-energy patch distributions (uniform digit backgrounds in SVHN) saturate the codebook with fewer samples.

### Empirical validation

| Dataset   | $\overline{\|\tilde{p}\|_2}$ | Predicted | Grid-search optimum | Error |
|-----------|------|-----------|---------------------|-------|
| CIFAR-10  | 0.5217 | 27 | 28 | $-1$ |
| SVHN      | 0.3082 | 16 | 15 | $+1$ |

Both predictions land within $\pm 1$ of the grid-search optimum using a single constant $c$ — a tolerance well below the noise floor of patch-norm estimation from random sampling. By contrast, pixel-based formulas (Otsu, percentile, $\sigma$ power law) **rank the two datasets in the wrong order**, predicting more patches for SVHN than for CIFAR-10 despite SVHN's lower optimum.

### Cost

Patch sampling already occurs inside `conditioned_init`. Computing $\overline{\|\tilde{p}\|_2}$ requires one additional `np.linalg.norm` call before the k-means fit — no architectural changes, no extra hyperparameter search.

### Open question

The constant $c$ is currently fit from color datasets only. Generalizing to grayscale (where patches are $9$-dimensional rather than $27$-dimensional) likely requires a channel-aware correction of the form $c = c_0 \cdot g(C_{\text{in}})$. Establishing the functional form of $g$ is left to future work.